# Analisis Spasial & Temporal Data Curah Hujan GSMaP (NetCDF)
Notebook ini digunakan untuk membaca, menggabungkan, dan menganalisis dataset NetCDF (`.nc`) GSMaP dari folder `data/gsmap/` yang mencakup rentang waktu 2005–2026.

In [ ]:
import os
import glob
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")
plt.rcParams['font.sans-serif'] = "DejaVu Sans"
plt.rcParams['font.family'] = "sans-serif"

## 1. Memuat & Menggabungkan File NetCDF GSMaP
Menggunakan `xarray.open_mfdataset` untuk memuat semua file `.nc` dari subfolder `data/gsmap/YYYY/` secara efisien.

In [ ]:
data_dir = Path("data/gsmap")
nc_files = sorted(list(data_dir.glob("*/*.nc")))
print(f"Total file NetCDF ditemukan: {len(nc_files)}")

# Buka seluruh dataset menggunakan xarray
ds = xr.open_mfdataset(nc_files, combine='by_coords')
print("\n--- Informasi Dataset GSMaP ---")
print(ds)

## 2. Ekstraksi Deret Waktu Rata-rata Wilayah (Areal Mean Time Series)
Merata-ratakan nilai curah hujan di seluruh grid spasial (dimensi `x` dan `y`) pada setiap jam.

In [ ]:
# Hitung rata-rata spasial (Areal Mean)
da_areal_mean = ds['precipitation'].mean(dim=['x', 'y']).to_series()
df_hourly = pd.DataFrame({'rainfall': da_areal_mean})
df_hourly.index.name = 'DateTime'

print("Statistik Data Jam-jaman (Hourly):")
print(df_hourly.describe())

## 3. Agregasi Waktu (Standar WMO - SUM)
Akumulasi volume curah hujan dari jam-jaman ke Harian (Daily), Bulanan (Monthly), dan Tahunan (Annual).

In [ ]:
# Resample menggunakan sum() sesuai aturan meteorologi WMO
df_daily = df_hourly.resample('D').sum()
df_monthly = df_hourly.resample('ME').sum()
df_annual = df_hourly.resample('YE').sum()

print(f"Total hari: {len(df_daily)}")
print(f"Total bulan: {len(df_monthly)}")
print(f"Total tahun: {len(df_annual)}")

## 4. Visualisasi Time Series Curah Hujan Harian & Bulanan

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(15, 10), sharex=False)

# Daily plot
axes[0].plot(df_daily.index, df_daily['rainfall'], color='navy', linewidth=0.8, alpha=0.8)
axes[0].set_title("Curah Hujan Harian Rata-rata Wilayah GSMaP (2005 - 2026)", fontsize=14, fontweight='bold')
axes[0].set_ylabel("Curah Hujan (mm/hari)", fontsize=12)
axes[0].grid(True, linestyle='--', alpha=0.6)

# Monthly plot
axes[1].bar(df_monthly.index, df_monthly['rainfall'], color='royalblue', width=20, alpha=0.85)
axes[1].set_title("Akumulasi Curah Hujan Bulanan GSMaP", fontsize=14, fontweight='bold')
axes[1].set_ylabel("Curah Hujan (mm/bulan)", fontsize=12)
axes[1].set_xlabel("Tahun", fontsize=12)
axes[1].grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

## 5. Visualisasi Distribusi Spasial (Peta Rata-rata & Total Curah Hujan)
Melihat sebaran curah hujan di seluruh piksel grid area studi.

In [ ]:
# Hitung total akumulasi curah hujan secara spasial
spatial_total = ds['precipitation'].sum(dim='time')
spatial_mean = ds['precipitation'].mean(dim='time')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Peta Total
p1 = spatial_total.plot(ax=axes[0], cmap='YlGnBu', cbar_kwargs={'label': 'Total Curah Hujan (mm)'})
axes[0].set_title("Peta Total Akumulasi Curah Hujan Spasial (GSMaP)", fontsize=13, fontweight='bold')
axes[0].set_xlabel("Bujur (Longitude)", fontsize=11)
axes[0].set_ylabel("Lintang (Latitude)", fontsize=11)

# Peta Rata-rata Jam-jaman
p2 = spatial_mean.plot(ax=axes[1], cmap='Blues', cbar_kwargs={'label': 'Rata-rata Intensitas (mm/jam)'})
axes[1].set_title("Peta Rata-rata Intensitas Curah Hujan Spasial (GSMaP)", fontsize=13, fontweight='bold')
axes[1].set_xlabel("Bujur (Longitude)", fontsize=11)
axes[1].set_ylabel("Lintang (Latitude)", fontsize=11)

plt.tight_layout()
plt.show()

## 6. Pola Klimatologi Bulanan
Melihat profil rata-rata curah hujan untuk tiap bulan (Januari - Desember) selama periode pengamatan.

In [ ]:
monthly_climatology = df_monthly.groupby(df_monthly.index.month).mean()

plt.figure(figsize=(10, 5))
bars = plt.bar(monthly_climatology.index, monthly_climatology['rainfall'], color='teal', alpha=0.8, edgecolor='black')
plt.title("Klimatologi Curah Hujan Bulanan GSMaP (Rata-rata 2005-2026)", fontsize=14, fontweight='bold')
plt.xlabel("Bulan", fontsize=12)
plt.ylabel("Curah Hujan (mm/bulan)", fontsize=12)
plt.xticks(range(1, 13), ['Jan', 'Feb', 'Mar', 'Apr', 'Mei', 'Jun', 'Jul', 'Agu', 'Sep', 'Okt', 'Nov', 'Des'])
plt.grid(True, axis='y', linestyle='--', alpha=0.7)

# Nilai di atas bar
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 5, f'{yval:.1f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()